# BatikCraft — Offline LoRA Training Notebook

Notebook standalone untuk melatih LoRA dari file `.batikdataset` hasil **Dataset Studio**. Notebook tidak memerlukan repository BatikCraftStudio. Hasil akhirnya adalah `.batikmodel` yang dapat dipasang ke aplikasi dan dipakai sepenuhnya offline.

> Training membutuhkan GPU. Inferensi di aplikasi tidak mengakses internet.


## 1. Tambahkan input

Tambahkan melalui **Add Input**:
- satu file `.batikdataset`;
- base model Diffusers lokal bila ingin training tanpa internet;
- opsional model lain yang dibutuhkan untuk validasi.


In [ ]:
from pathlib import Path
import base64, hashlib, importlib, subprocess, sys, zlib

# Install training dependencies only inside Kaggle/local training environment.
packages = [
    "diffusers==0.39.0",
    "transformers>=4.48,<5",
    "accelerate>=1.2",
    "peft>=0.17",
    "safetensors>=0.4",
    "datasets>=3.2",
    "torchvision",
]
missing = []
for module, package in {
    "diffusers": packages[0], "transformers": packages[1],
    "accelerate": packages[2], "peft": packages[3],
    "safetensors": packages[4], "datasets": packages[5],
    "torchvision": packages[6],
}.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

PIPELINE_SHA256 = "741748ad9b741127c30c3c7a25a684160fd4fe660655ca1c27a616c0d2d182cb"
PIPELINE_B64 = (
    "eNqlPP1v20ayv+uv4CPwAOpKM7bvkmsFsHhuPnpB016Q+O4Bz89g1uRK4okieeTSjprn//3NzH6TlOK2BWJJu7Ozs7MzszOzsw3D8KNgdcGqpu"
    "bBD0yUu5cdW4vgXfPhKhAdK+uy3gRt2fKqBIh10wU/sc2m4gF8Y0HV5KwKfnz/j2SxuN6WfbAuoasA4DveMcGrQ7BlfVA3Qblvm04AQNfsg093"
    "OFOOM2W9GIqy+ZQEVwAm+F3T7IKc1QHf3/EiKEUA5ElKFjQWCf2UEIKCCdZz8SkOxJbDiM84BQGUdS9YVbG7ygDvGyDrU7IIw3AhMWXZehBDx7"
    "NMUQdTAQkA3NT9YqHagP4tLEf//Fff1Pr7nomt/t4Bmc1e/+q3gygr/evXskW+yFmR6Lxifc97M21flLmIbZeEbAE9zKyh3uNs1CEOLe6Kar+q"
    "D4vFD1fXb3/KXl1dX318fZ29+fuHn6+ugzQILaPP9G6eKbaFatDPf3/1+t38EOLZWcvyHfBs8V+GvqivGtGn193AlwtqCa4V9pdNvS43q0UA/6"
    "mJMlzIKuhFR6130JYRZtvWDKIdRFaUHbUhGc92JGfPHppuB3ifOWRVTcfO5JCQhhO2rCzMYAc4H3rR7M/uL1zQmu25AXbE/iUBk/RLcNGVmw3v"
    "MqDCwZ53Gc0QjhaUrdm+rA4GsC8unkuYjvdNNaBkrUA4BXQ+v7hUMwDjEF2+zfryV677L6h307Gi5LXIWJ4P+6Ei4QSd4W2vAf8i18U+ZxKV13"
    "lxeX5O/RVnHe5Phmq5CtZVw6ifn8nxIL87M+gFNfWcF7rp8vxSNu7Lz7zI2o7nZU+rUUtdtxcv5FLzLc93bQPjcDqPmsvnkph7VpWFXEoLAt2K"
    "KWtjMDC4SzXocfAWdKvmfQlmgTphF0W5Dse4yj3b8BFfgNBmD2gKoPqBl5utsIs/T759TkBsENvGip5EXPA+78pWuKuUPVWZ8xr23JMi1JCCr8"
    "EKwTbkInMtVCRn6fJtec+tOgT/R0odz6qA7lsGZ98HYmgrfoNW4gb6YtT52xjo6MWo8fZWqh5YuX9KxnC5If2w78mU5k19z8FsCNZtuOif5YyW"
    "2AeiCd4iA980VcG7YM8FQ/ITMpgkD83Q5RzWioRF7mqWmmGw5bQVGsguSoKUaxcq4Z9hBX20lDTTHGQ4k24vOs4jB1aNpx1GfDCDO9+zICTpD0"
    "dgyX4Hf6OWdSBJ2mIhyEMpttowJ/9Ttm/gM5IrjIOwC5dglfWOWfL2rC7XMC3MjgdBAoJU9JoVScdZEYUaJkGIcJkUIIEFj8JBrM++DZdLgwyY"
    "YWBhK6IQTlc4UmDq/0iDOYNu6ZAaW/Y8+DCAnu35665ruijEVQR3ww4UxbFq5hxX4piEloie7UG0eliQT4tqDz1y4XQMQO3xcK1zYJeEkYK4RI"
    "8AAVTr14l9JakJBIgpqDQH01nuSjXepVGLYtY1D6DeI7GHP7e3QP/NrRlQIyMrMKbFalZJfGh0akowEJ9jNTf8Cng97MmF0atc+gtCuUGuKWrl"
    "/mFbGAdfHpcerFVAH77fssvnL2YGSN0k+wJDCK0cITtCH3q6Mc544s+I9iMbsg4/yuV/IW4E3wQXj+PNkZiTEQVgUwQno+upgkPGhGLlViWSB5"
    "HCsEy2/HNRgvKKiNTAsM5f/xPX81KNVmTr7Z2uL2/yZjdeFfJdb0Eox6zOXxSPSVtvQg8ycuzSMzNumTx0peDZ3UHw3qzQZ5w0vigXootc2VA9"
    "IBwh2IwEuss2mt13BflUlqgZZ1ixa8BgbcZc8JQvYW0LZ2n0hWSddjZcmQUDrQKOP2hRNI3E2qqlxjOh+cukhU6z2dnmQTXjDBFH4QTfNN0BAH"
    "2+q2Zi/JGxvThUfDRQtp0cRadLxu5ZSZEJIFBtIVocUvNjxDZ1UeJi0KXyELg9X0WzZ/3OH44tp4Y9ei3O0emLfGi8BTz0Kjj1GtjfKHwAhvAa"
    "zj+gLtUnIJ6sW3BFKudgRRsMAoaU+Nbem1+OkmoV0QFcDPu2jwAS5+kxmGN9XpbpG1b1fAlyHf5vrQS64xQupY6MhToUWpnT78a03VpuqNMwyx"
    "tQJgAGAYysMC8ngD3A2H7ZLTkZjZwWNdmZJE67DNJuoC5ZDXGXS8BzK47pIKtFeukQNd4A2aOZAhFwPbt6ZwXKtZXxBUZfaM0wzBuFfeSrouNn"
    "nFDqD/7xC1cpBemC91KGXpXr9dDzDjyI96/fXMfkn17lOa/o5LWeZ4ahSMbBbz2ILcympk+weTnynrXrqUDGHihELugMFgrUGQe7oTttiOyGsV"
    "owY8dpOu7tq3MRaXDjYLstLiXulrghJ8ygcHitoCxjotT56EChLbKR6phNM44xCBT641mzU36y8qVJbUQDJ/ukIanrZD3UOYozxGmg2PYXAVO+"
    "gpktNTkL3dJ0FkwtyaRF0LnOVKMDpYXG4BpEQyLOu5/excGrV+9//ghuQzHABHHwUaCpk5IGZL1XmayYZPLy1UttPn/G+H1mFjBkcHyWv0qNVV"
    "OiV9PrOeYGYRBjCFQRF4TCwIEMXdFMNJkBtuNbDs66GvQOtEwqld/vo0ZCsFWlH+wMdpDcJxpFgqiHovf9DjjsLoBg7ymwN9vcsbrHwMSh07Q5"
    "u/Dy3dv316ANxMdY/mx2vAbz0UlBYnbPQawdCbD6ciLbkSo9OAFiVWuUpNBjR82e0mlagTSP9gRXjGPI+PFiotw2+2Pn74e7NUXRaWgQe2YXTX"
    "umhFbPqJn3h2d0cHuT3jN0ZT11mcw1ncJFDRiUIRlqjkfpnBL9RpyISSGtG3BYrWIBfk+Z/yhjDGLDFc0WCFf+PZQduDMoXlkkvYfJTp0Aw1V8"
    "rZsVRQa2uhXcEXmr574z3GmhxcSc75fhAZyxqt2y4zBwKAs6qVXSq0/DDRt6TKGF8VysCWwD1vTpDUhstsOzAz7/rT7v1SccH8m56xgtXUYqUw"
    "3b5lpuu6yQXEa5Gw4R5OvBWZRiCOQdzyap4zgzfVuVIlXtrnQT8syYJqDCmq7kZQNWqueWlhufBxbyA8ck7DQw0Zw2edypr1xCgNe1jTRJqYPz"
    "rduBOpL88Pbd219eX33wkYwCB5d+jihedk0bTQg5PuoD3Un8renKXxvMo76pIIZs0/Pk4vnxQdfNNfiVYJWPg/yiHcPo5jx5Dp4ifTjwt55YgO"
    "eI29o1Oe/7iH9W+aHAJmUoR0OZGfIg/WyNDQFkyIHJG/qWqIM1Cj/8+AOEFZTHwQ7K36hpbqTUhbc232MynulIDDLlUCsvKkKBRP1fxp5TNsoS"
    "wKwI5E9KcfDtiCHkjBuQFk6jKrtn1QDxwq1ZlRVhGWKN1iV5MIevrNG1KwuJzJw70VzCYRTp4eUBxDQbsU3NuES6FLbLH9KCNaOYwgKMDUsHrq"
    "DUBfIuffNGcQeEOChqYJxa4YxeJmYtC3+AWe7Y4ujMJsYVDget3DnCmDdVhYk9K4lzOcKTkujuHTEbfSzwvvJddKPQTjYYt1H1ubJy62RY9aqP"
    "onT3+Ov4FM/8nIpP1spbSiKaaM/3TXfIZC46lWRg7qrcDM3Qq/ZlQncoYxvh0Leyq7Ewj3ITKnI78Xg3PmjkHQewk7F7JbBeV3wkRGoTszV6d/"
    "TV9tnLtNRET/4lm4Wthz0q9Q5c2fTSD8NgCF0gg2bCmcQg0gKikev2F/CdjnbT0kdLzMuZ375PINVWRRTEAclfakmuCrb/b8sIQ4AltjJegXed"
    "56wcosE+jc6T7+IA/nz3nbNB0hHICp6zQ3rBzy6dDAH419Dyrbv+ocXLox7Ciy7jbZNv6YpAgDDwsrJUYipE7ucSzuuvO+nODIS199BqR92/zA"
    "TE0BJNKIqDC3Wd4vqMXnDmeB+AG9PjIiMb8cC6/dA6RsfsSmq+jaSERqhoZF/W0cX5eRwco/hZcHG+9BHoaxg/oBmN9LYARCu2hMVKcWJvuU5Y"
    "lSjvyS6aMEyX6EiUxGiVzUTODh1acsShRW1wRJSkl0yBujNWeWmXqFHUFaTebTEJAvjJCtfd70B2tz6CjHD9+dIPgdDGuRgLfl/inR+tLnWXOh"
    "MA/Maxm6q5YxVtLF43K+OHpR/qzCEK1XXUQnsTSt3AtsA5toFzivTEufMgk0My49wI4EgycDhQbqufPKWUm0u9UU4eIcaZSxXsqwWqKDJOMiGa"
    "TS3TzOMjD9k1Cz1KB2g+HoWd8ncedjlpXSZyDRBP9CKRebtoeWKh+tufaMk6z5iDuwuKu2a5zlj59xp40aKNOVYC1VlV7nikkE3nsyeRnRKvwl"
    "p+c347gcb7G2lW3Dkg3Jjn7vk8c0bRtV6bsUyZmWZ+fOQcn0euOOQ+pnpBx7YVNqWpNzPbgCQeMrsZY5Ixgqa2+YWrgSeWP991YuFTGpUpyLZl"
    "UfBaptxoYxw7cVJHHB9u3p7MSPGcVIARRPdUXh+iAkdH1214+pvXf3TFc7sq1WsqvvLy1T04Tsklnt/3vGryUhy0BsVq+yypy1lcdCU6K+aWW5"
    "k8xuDguM9sazi/dDxMCOMTJAMtO94Vmwx4sgdKsHV+4XbyeW/a599pGEA1ECoIyPgkxzNPryt5dxBngINTREjuFHR0CveHOs+0k9evFl+z73lV"
    "tjIphndKWWRd2+AiOZ9OZxyVBPd6xlDY7T0CYDHAv4bmjvC6RDRAQs2dQqDfuUT3aP9GV+uNpYGbC3O5edgGSg5B3TaCIFfwfbRcznJ7XmKULM"
    "9U2AXfK/disg114dH6n6eQpOkMluXX97dn91zahePnPSZTvNu2dWhJOPvi0PgYLp9wugOT3HV9nx5xxafU33Wc7RZ/DI/EMb7XSB5YKTBElneV"
    "IGiRqXtzwco+2yNilZtwPbuHDiWmGDn2sl3mtqWztvCywcR7eVl46qrJ352jl0eRIWO5mDL/yI2aFAI35ezPRt2w8xxdqEM6EYcQh45rJ3CpEm"
    "fFDhii29XGI+xrkEDelZiXnEs6uVmRSR2Ajg9h28OVKsdWseg4w6HvXWcqAnSh2GiEI10A6PwawaHdptMC1tqLDKJLAJ/qUz/sI2lbbs4AZnWL"
    "cbeMwr1GEDnZIg+xX0AYTyWb/avjlZfwdBM4xhv0N88Ul3+9kOGPFTMcL2iwu/wUjVS5sRkRpEKAA/nZnjgnKGMqYamLfTe8pnvtzKtoBi+OPy"
    "jxny+UiBdGdafVwE7FLzUESZLYst4f1ZTgbAve7QFpL8rcKYMONAEBW2OOSm+NLaeYXOkfv1jX2v2m6fCC8pIqhHUqFZ1VLH3OhwLYBgKnsoXw"
    "E02cKTSKllIGw7wdVEmFSiV4CQRZHSyRGqxHwnnzIiU9Tubvui+U+y4DTifSXKKr7rrnupBFVZuYrVx65FGB8BGjiBZQi5Ysw5gm67DOLdVAft"
    "GbukmUIM6DB5Nm8m7HDEFou9RQh5abWQS309n0beKNuYYal9ffetMqUTxdmKNKbwgynIx7SqWKrW3RmRWcyqnzlc8M4Pf4jkdRM3mREM8V3iwX"
    "Xr2wTdFMsch7Gsd3UqaisVnfH3WLEqxUyRd4HvWAZwS3l+f4PfhGTrv0L8QAn95d38iqhfglmBDtl/WaAy9zrhKRfx4lDDYDLIK6c1bx9K/JuF"
    "/TnZpv3t0NrdyNWFEvkEpXGND/Uw1ntgh1dX45rq+VV33oPURWv5z91h627VSmnSyoOqbw+ocM9t1QVkVmH4T9biPtSulqYq0x2F0qS+7Vw72H"
    "OAv3TJkkbUZig0sWwSkfhfXuczf1gs2Y8SfXvKlJMrUPcwZLTqghnnC2L0wRvFI4yQJdkR8HVPUsS+2t00An7LOvnq6xOopjj3QqKlfV1Cpqul"
    "2MCvgVG/0Kfstb8Bw7kV44aimL65UMRUYm+2e+cFrBjI2tCs10PmXG5XeYiuchThUdnXqezfGE/dDk4p2ZmYqlnZcqlv/jm0BAgDeAbHy5G3YN"
    "lQfjx6hHvVlYPaWAfzwU4nHlKmvoOc9S3rChJtH05nGBLlJ2Nt5ZpFPUq57RrILp+0a3Uheizz3LwCvsZbl4eJGcO65kSOoG7SOmlegd6xBRvT"
    "scLVQVqntA04r10J/am1x65OB9YKdUBb9v8ubQzjfpOuHo4z3tzcxBdzuWh8kxb+eb9k0G61IVd9B8HQ2W77O7sipFSbfTN5P4J2zu/gUx5EyG"
    "K+zB65fJvJnOsm6ZTHbMDhXdkONj4EI/LPWBxgwh04XiqDfoCUbNxyDfHFqOyN/jaNO+QbSQTuOYKvkw0YK6LxXjSbgrQMEwwDYSFKITwbrx4s"
    "Oab+AAwsd+5FLMxqVhu22AA5xVFJCA6WYYn7BuF1NuHE6Eaui6QxyAVd7A0Sf3EY1yA15TgBnITQfhdBGejFN1wfNEM5Vw47mKbHXfAPfmjbt8"
    "O3x/MS8+3NFt/DUrKOAyOco9d3vrSz+rd47cTyr1Hh0b6Fgf+ZZs5Rtz9yWB/9pylFn7MjJQj44HER5//2iLMCxyp6oWRqf4pMOJo/ZwAvZUY2"
    "twvX2fvXr95t3V9etXU8gKwvAqfaFihNk3lvoJGeUOMEga7b/3vDI+llYwLDudWNCXmvptylzCyB5I2fyJ5KcGJ/TLsd77L+WiOmxW/ukkPGH2"
    "AXI8fY1ODiZ8rsxVNSYB5RjvzZj2ialQbe2/MShle0KllTpWBxlygR7j4AsCPerMx+RJhnktTjRhummlH5YXzZ7UKbLvNUR3cOv6yNsFf609oE"
    "jU7cIWTrTJPAb+OecQ0b2lsfS6zSnPwv+HwmKRZayqssy4QKHv3yvhCccxgW6fe9uh+04lfjSMfS4DLbeL/weDsWos"
)
source = zlib.decompress(base64.b64decode(PIPELINE_B64)).decode('utf-8')
assert hashlib.sha256(source.encode('utf-8')).hexdigest() == PIPELINE_SHA256
pipeline_path = Path("/kaggle/working/offline_lora_training_pipeline.py")
pipeline_path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(pipeline_path.parent))
from offline_lora_training_pipeline import (
    TrainingConfig, build_batikmodel, generate_validation_previews, train_lora
)
print("Standalone training pipeline ready:", pipeline_path)


In [ ]:
from pathlib import Path

DATASET_OVERRIDE = None  # contoh: '/kaggle/input/my-data/wayang.batikdataset'
candidates = list(Path('/kaggle/input').rglob('*.batikdataset'))
DATASET_PATH = Path(DATASET_OVERRIDE) if DATASET_OVERRIDE else (candidates[0] if candidates else None)
if DATASET_PATH is None or not DATASET_PATH.is_file():
    raise FileNotFoundError('Tambahkan satu file .batikdataset melalui Add Input.')
print('Dataset:', DATASET_PATH)


## 2. Konfigurasi

`BASE_MODEL` boleh berupa folder lokal Diffusers di `/kaggle/input/...` atau model ID bila internet Kaggle diaktifkan. Untuk paket aplikasi offline, salin base model yang sama ke komputer user.


In [ ]:
BASE_MODEL = '/kaggle/input/stable-diffusion-v1-5-diffusers'  # ubah sesuai input

config = TrainingConfig(
    dataset_path=str(DATASET_PATH),
    base_model=BASE_MODEL,
    output_dir='/kaggle/working/batikcraft-lora-output',
    model_id='batikcraft-wayang-v1',
    model_name='BatikCraft Wayang v1',
    trigger_word='bcr_wayang',
    base_model_family='sd15',
    resolution=512,
    train_batch_size=1,
    gradient_accumulation_steps=4,
    max_train_steps=1200,
    learning_rate=1e-4,
    rank=16,
    seed=2026,
    mixed_precision='fp16',
    checkpointing_steps=250,
    validation_prompt='bcr_wayang, tokoh wayang kulit dengan ornamen batik klasik Jawa',
    validation_images=4,
    recommended_weight=0.85,
    author='Balya Rochmadi',
)
config


In [ ]:
lora_path = train_lora(config)
print('LoRA:', lora_path)


In [ ]:
# Preview dapat dilewati bila VRAM terbatas.
previews = generate_validation_previews(config, lora_path)
previews


In [ ]:
model_pack = build_batikmodel(config, lora_path, previews)
print('Install this file in BatikCraft Studio:', model_pack)


## 3. Pasang ke aplikasi

Download file `.batikmodel`, lalu buka **AI Batik → Kelola Model Offline → Pasang .batikmodel**. Pilih folder base model dan ControlNet yang telah tersedia di komputer user. Sesudah aktivasi, gambar tidak dikirim ke internet.
